# Model 01 — Logistic Regression (Logistics Delay Risk)

This notebook presents a step-by-step analytical walkthrough of a logistics delay prediction problem using Logistic Regression.

The starting point for this analysis came from a recurring operational discussion. During a logistics performance review, an operations engineer asked whether delivery delays could be anticipated early enough to support proactive decision-making, rather than reacting once delays had already materialized.

After exploring the nature of the problem, Logistic Regression emerged as a natural first tool. Not to predict exact delay times, but to quantify how operational factors translate into **delay probability** in a transparent and interpretable way.

Data source: `logistics_shipments_data.csv`  
Output: metrics + coefficient interpretation + probability maps + scenario simulator

## 1. Setup

This section defines the environment used throughout the notebook. The goal is reproducibility: clear imports, consistent random seeds, and a small set of standard libraries commonly used in operational analytics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

## 2. Load data

We load the logistics dataset directly from a CSV file (as it would typically be exported from operational systems). Each row represents a shipment record with contextual and process-related variables, plus a binary label indicating whether the shipment was delayed.

In [ ]:
# Load the dataset. No synthetic generation here — we load the CSV like we would in real projects.

df = pd.read_csv("logistics_shipments_data.csv")
df.head()

## 3. Quick sanity checks

Before any modeling, we validate the dataset at a basic level: dimensions, missing values, data types, and target balance. This prevents silent issues from propagating into the analysis.

In [ ]:
# Sanity checks. We check shape, missing values, distributions, and class balance.
# This is where real datasets usually try to hurt you 🙂

df.info()
df.isna().sum()
df["Delay"].value_counts(normalize=True)
df["Departure_shift"].value_counts()

## 4. Exploratory data analysis (EDA)

The purpose of EDA is to understand how delay risk relates to operational drivers. We focus on distributions, group comparisons, and simple visual checks that help form testable hypotheses before training a model.

In [ ]:
df.describe()

In [ ]:
# Exploratory Data Analysis (EDA). We want a feel for the operation: where time gets lost and why.

df.hist(figsize=(12,10))

## 5. Preprocessing

Logistic Regression requires well-prepared inputs. In this section we:
- split data into train/test sets,
- encode categorical variables,
- and standardize numeric features (useful for stable training and comparable coefficients).

Key preprocessing steps:
- One-hot encoding for categorical variables
- Train/test split
- Standard scaling for numeric features

In [ ]:
# 1) One-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

# 2) Split
X = df_encoded.drop("Delay", axis=1)
y = df_encoded["Delay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3) Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_names = X.columns # relevant to create a simulator!

## 6. Train model

We train a Logistic Regression classifier as an interpretable baseline. The goal is not only performance, but a model that can be explained in terms of probability and operational impact.

In [ ]:
# Train Logistic Regression. This model is simple on purpose.
# In operations, *interpretability beats fancy* (most of the time).

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

## 7. Evaluate

We evaluate the model using standard classification metrics. More importantly, we inspect error types (false positives vs false negatives) and how well predicted probabilities separate delayed vs non-delayed shipments.

In [ ]:
# Evaluation. We care about more than accuracy: 
# precision/recall tells us how the model behaves under pressure.

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# ROC
y_prob = model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

## 8. Coefficients as operational drivers

One advantage of Logistic Regression is interpretability. Coefficients can be translated into odds ratios, enabling a practical discussion about which factors increase or decrease delay probability.

In [ ]:
# What drives delay risk? Coefficients tell us direction:
# - positive → more delay risk
# - negative → less delay risk

coef = pd.DataFrame({"Feature": feature_names, "Coef": model.coef_[0]}).sort_values("Coef", ascending=False)
coef

In [ ]:
# Plot

coef_sorted = coef.sort_values("Coef", ascending=True)
plt.figure(figsize=(6,4))
plt.barh(coef_sorted["Feature"], coef_sorted["Coef"])
plt.axvline(0, linewidth=1)
plt.title("Feature impact on delay risk (log-odds)")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Summary of practical signals (example interpretation)

After training, several patterns typically stand out:
- Urban routes tend to increase delay risk (traffic variability).
- More stops compound operational complexity.
- Longer loading and waiting times reflect congestion effects.
- Night departures may introduce higher uncertainty (handoffs, staffing, constraints).
- Operator experience and shipment priority often act as protective factors.

The exact ranking depends on the data, but the workflow remains the same: quantify, interpret, and validate.

## 9. Probability factors and correlations

In this section we visualize how key variables relate to predicted delay probability and explore correlations that can explain why certain drivers move together (and how that affects interpretation).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

coef_df = coef.copy()
coef_df["Odds_ratio"] = np.exp(coef_df["Coef"])

coef_df = coef_df.sort_values("Odds_ratio")

plt.figure(figsize=(6,4))
plt.barh(coef_df["Feature"], coef_df["Odds_ratio"])
plt.axvline(1, linestyle="--", color="black")
plt.xlabel("Odds Ratio")
plt.title("Impact of operational variables on delay risk")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
sns.heatmap(df_encoded.corr(), cmap="coolwarm", annot=False)
plt.title("Heat Map Correlations")
plt.show()

## 10. Simulator (what-if scenarios)

A simple simulator turns the model into a decision-support tool. Instead of only scoring the test set, we can define hypothetical shipment conditions and estimate the resulting delay probability. This is the most practical way to connect a probabilistic model to operational conversations.

In [ ]:
# Predicted probabilities on the test set
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Display the first 10 predictions
for i in range(10):
    print(
        f"Delay probability: {y_prob[i]:.3f}  |  "
        f"Predicted class: {y_pred[i]}  |  "
        f"Actual: {y_test.iloc[i]}"
    )

# Note: probabilities close to 0.5 represent higher uncertainty

In [ ]:
import numpy as np
import pandas as pd

def simulate_shipment(
    distance_km: float,
    weight_kg: float,
    number_of_stops: int,
    shipment_priority: int,
    loading_time_min: float,
    dock_waiting_time_min: float,
    operator_experience_years: float,
    route_type: str,
    departure_shift: str,
    threshold: float = 0.5
):
    """
    Simulates a shipment scenario and returns the delay probability
    along with the predicted class.

    Parameters
    ----------
    route_type : 'urban', 'mixed', or 'long'
    departure_shift : 'day' or 'night'
    threshold : float between 0 and 1 used to classify delay / no delay
    """

    # 1) Build a single-row DataFrame in the SAME format as the original dataset
    new_shipment = pd.DataFrame({
        "Distance_km": [distance_km],
        "Weight_kg": [weight_kg],
        "Number_of_stops": [number_of_stops],
        "Shipment_priority": [shipment_priority],
        "Loading_time_min": [loading_time_min],
        "Dock_waiting_time_min": [dock_waiting_time_min],
        "Operator_experience_years": [operator_experience_years],
        "Route_type": [route_type],
        "Departure_shift": [departure_shift]
    })

    # 2) Apply the same one-hot encoding used during training
    new_shipment_encoded = pd.get_dummies(new_shipment, drop_first=True)

    # 3) Align columns with the training feature set
    #    Missing columns are filled with zeros
    X_new = new_shipment_encoded.reindex(columns=feature_names, fill_value=0)

    # 4) Scale using the same scaler fitted during training
    X_new_scaled = scaler.transform(X_new)

    # 5) Predict probability and class
    prob = model.predict_proba(X_new_scaled)[0, 1]
    predicted_class = int(prob >= threshold)

    # 6) Print interpretation
    print("=== Shipment Simulation ===")
    print(new_shipment.to_string(index=False))
    print("\nModel output:")
    print(f"Delay probability: {prob:.3f} ({prob*100:.1f}%)")
    print(f"Decision threshold: {threshold:.2f}")

    if predicted_class == 1:
        print("Model classification: DELAYED (1)")
    else:
        print("Model classification: ON-TIME (0)")

    return prob, predicted_class

In [ ]:
simulate_shipment(
    distance_km=120,
    weight_kg=500,
    number_of_stops=1,
    shipment_priority=1,            # urgent shipment
    loading_time_min=40,
    dock_waiting_time_min=5,
    operator_experience_years=10,
    route_type="long",
    departure_shift="night",        # night shifts are often smoother for long-distance routes
    threshold=0.5
)

In [ ]:
simulate_shipment(
    distance_km=50,
    weight_kg=2500,
    number_of_stops=4,
    shipment_priority=0,            # standard shipment
    loading_time_min=110,
    dock_waiting_time_min=90,
    operator_experience_years=1,
    route_type="urban",
    departure_shift="day",
    threshold=0.5
)

## Final Reflection

This project is not about predicting the future with certainty.  
It is about **making uncertainty visible**.

Logistic Regression works especially well in operational problems because it forces us to think in probabilities, trade-offs, and risk — the same way real-life decisions are made on the shop floor or in a control room.

There is no single variable that “causes” a delay.  
Delays emerge from **process interactions**: congestion at the dock, loading times, route complexity, experience, and timing.

What this model offers is not a verdict, but a **conversation**:
- *What happens if waiting time increases?*
- *How much risk are we adding by routing through urban areas during the day?*
- *Which levers are actually worth improving?*

The simulator reinforces this idea.  
Same model. Different scenarios. Different risks.

And that is the real value of Logistic Regression in operations:
not magic, not certainty,  
just probabilities that help us ask better questions and improve the process step by step.

—
Not magic. Just probabilities.  
**Where f(x) meets Kaizen**  
**LozanoLsa**